In [1]:
import nltk

In [34]:
#pip install bert-score

In [35]:
#pip install rouge_score

In [13]:
import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [14]:
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score

import pandas as pd

In [15]:
#Load data
df = pd.read_csv("ai_teacher_results.csv")

In [16]:
#preparing references and genrated answers

references = (df["Teacher Response"].fillna("").astype(str).tolist())
candidates = (df["GeneratedAnswer"].fillna("").astype(str).tolist())

In [24]:
#BERT score

P, R, F1   = bert_score(candidates, references, lang = "en", verbose = True)
df["BERT"] = F1.cpu().numpy()

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 222.03 seconds, 0.09 sentences/sec


In [25]:
#ROUGE

rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer = True)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for candidate, reference in zip(candidates, references):
    scores = rouge.score(candidate, reference)

    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

df["ROUGE-1"] = rouge1_scores
df["ROUGE-2"] = rouge2_scores
df["ROUGE-L"] = rougeL_scores

In [26]:
#METEOR
meteor_scores = []

for candidate, reference in zip(candidates, references):
    candidate_tokens = candidate.split()
    reference_tokens = reference.split()
    score = meteor_score([reference_tokens], candidate_tokens)
    meteor_scores.append(score)

df["METEOR"] = meteor_scores

In [27]:
# Round up the scores

similarity_columns = ["BERT", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR"]

# Only round columns that exist in the DataFrame
existing_columns = [col for col in similarity_columns if col in df.columns]
df[existing_columns] = df[existing_columns].round(3)

In [28]:
# QUESTION-WISE SIMILARITY DATAFRAME

requested_columns = ["User Query", "BERT", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR"]
existing_columns = [col for col in requested_columns if col in df.columns]

similarity_result = df[existing_columns].copy()

In [29]:
display(similarity_result)

,User Query,BERT,ROUGE-1,ROUGE-2,ROUGE-L,METEOR
0,What is AI?,0.884,0.275,0.077,0.200,0.173
1,How does AI differ from traditional programming?,0.885,0.293,0.100,0.220,0.331
2,What is Machine Learning?,0.893,0.400,0.241,0.376,0.423
3,What is Deep Learning?,0.907,0.380,0.156,0.329,0.424
4,"What is the difference between AI, ML, and Dee...",0.885,0.483,0.188,0.391,0.359
5,What is training data?,0.886,0.364,0.140,0.227,0.367
6,What is a Machine Learning model?,0.898,0.442,0.160,0.390,0.450
7,What is supervised learning?,0.870,0.289,0.049,0.241,0.310
8,What is unsupervised learning?,0.880,0.241,0.025,0.193,0.219
9,What is reinforcement learning?,0.913,0.338,0.116,0.197,0.425


In [30]:
# Calculating averages

average_bert = df["BERT"].mean()
average_rouge1 = df["ROUGE-1"].mean()
average_rouge2 = df["ROUGE-2"].mean()
average_rougeL = df["ROUGE-L"].mean()
average_meteor = df["METEOR"].mean()

In [31]:
# Creating average Similarity dataframe

average_similarity = pd.DataFrame({
    "Metric": ["BERT", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR"],
    "Average Score": [average_bert, average_rouge1, average_rouge2, average_rougeL, average_meteor]
})

# ROUND AVERAGE SCORES
average_similarity["Average Score"] = (average_similarity["Average Score"].round(3))

# DISPLAY AVERAGE DATAFRAME
display(average_similarity)

,Metric,Average Score
0,BERT,0.873
1,ROUGE-1,0.293
2,ROUGE-2,0.109
3,ROUGE-L,0.231
4,METEOR,0.295


In [32]:
#saving the calculations
df.to_csv("ai_teacher_evaluation_Similarity_metrics.csv", index=False, encoding="utf-8-sig")

In [33]:
average_similarity.to_csv("ai_teacher_evaluation_Similarity_Average.csv", index=False, encoding="utf-8-sig")